# Basic dataset exploration

## vaspout.h5 file

an HDF5 file.

In [ ]:
import h5py

In [3]:
! pwd

/c/Users/Libero/uni/1B/LCPB_2026/Project


In [12]:
f = h5py.File('./vaspout.h5','r')

In [62]:
#f.visit(print)

In [8]:
print(f['results/positions/position_ions'].shape)

(100, 3)


so only final step (100 atoms and 3 coordinates), no trajectories in this file. Probably bc of corruption.

## XDATCAR file

A plain text file. Includes trajectories only.

From here on we use the ASE default reader.

In [9]:
from ase.io import read

In [33]:
traj_XDAT = read('XDATCAR', index=':')  # list of Atoms objects

In [34]:
print(len(traj_XDAT))

print(traj_XDAT[0].get_positions().shape)

5406
(100, 3)


...so, same issue as before.

## OUTCAR file

A plain text file. Includes trajectories and everything else.

In [21]:
traj_OUT = read('OUTCAR', index=':') #---> note this
print(len(traj_OUT))

5406


...so, 5406 simulation steps: ok

MACE needs positions + energies + forces + (optionally) stresses per frame to learn the potential energy surface:

In [40]:
print(traj_OUT[0], '\n')
print(type(traj_OUT[0]))

Atoms(symbols='C16H56I16N8Sn4', pbc=True, cell=[[8.742892963, 0.0, -0.031376383], [0.0, 8.422714245, 0.0], [-3.957950916, 0.0, 20.036876932]], calculator=SinglePointDFTCalculator(...)) 

<class 'ase.atoms.Atoms'>


...THIS is what is saved with ASE.py and fed to MACE.

In [30]:
print(traj_OUT[0].get_chemical_symbols())
#---> positions at a given instant

['Sn', 'Sn', 'Sn', 'Sn', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C']


In [61]:
print(traj_OUT[0].get_positions()[:5])

[[ 1.2176   6.00568  5.07039]
 [ 3.5857   2.41703 14.93504]
 [ 5.5638   1.79433  4.91661]
 [-0.77886  6.62528 15.08889]
 [ 0.93053  5.43875  8.18366]]


In [60]:
#list(zip(traj_OUT[0].get_chemical_symbols(), traj_OUT[0].positions))

In [80]:
#print(traj_OUT[0].get_forces(), '\n')

print(len(traj_OUT[0].get_forces()))
#---> force on each atom (ion)

100


In [82]:
print(traj_OUT[0].get_total_energy()) #--> or potential_emergy. Per simulation frame.

-483.00793683


In [51]:
print(traj_OUT[0].get_stress())

[ 2.04159764e-05  2.66939983e-03  6.22910101e-03  1.86302806e-04
 -2.00454180e-03  8.24004035e-05]


...force per area on simulation cell: 6-component Voigt (xx, yy, zz, yz, xz, xy)

## POSCAR file

A plain text file. Initial structure reference.

In [22]:
atoms = read('POSCAR')
print(atoms.get_chemical_symbols())

['Sn', 'Sn', 'Sn', 'Sn', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C']


...(same order as positions in OUTCAR file.)

 ## Other useful context files

* INCAR ---> MD settings (temperature, timestep).

* KPOINTS ---> k-point mesh used.

* POTCAR ---> pseudopotentials info.

In [83]:
! cat INCAR

ADDGRID= .TRUE.        (Increase grid, helps GGA convergence)
EDIFF  =  1E-04        (SCF energy convergence, in eV)
EDIFFG = -5E-02        (Ionic convergence, eV/AA)
ENCUT = 400
Global Parameters
IBRION = 0
ISIF   =  3            (Stress/relaxation: 2-Ions, 3-Shape/Ions/V, 4-Shape/Ions)
ISMEAR =  0            (Gaussian smearing, metals:1)
ISTART =  1            (Read existing wavefunction, if there)
ISYM =  0            (Symmetry: 0=none, 2=GGA, 3=hybrids)
IVDW = 12 
KPAR = 1 
LASPH  = .TRUE.        (Give more accurate total energies and band structure calculations)
LCHARG = .TRUE.        (Write CHGCAR or not)
LREAL  = .FALSE.       (Projection operators: automatic)
LWAVE  = .TRUE.        (Write WAVECAR or not)
MDALGO = 3
NELM   =  90           (Max electronic SCF steps)
NELMIN =  6            (Min electronic SCF steps)
NSW = 10000
POMASS = 118.71 126.90 14.00  3.00 12.01
POTIM = 0.5  
PREC   =  Accurate   (Precision level: Normal or Accurate, set Accurate when perform structure latti

**Rough explanation**

TEBEG=300 — 300K, NVT ensemble (Langevin)

MDALGO=3 — Langevin thermostat

POTIM=0.5 — 0.5 fs timestep

NSW=10000 — max 10000 steps

ISIF=3 — cell shape/volume allowed to change → stress data matters

ENCUT=400 — relevant for MACE validation (plane-wave cutoff)

POMASS — atom masses: Sn(118.71), I(126.90), N(14.00), ?, C(12.01) — order matches POSCAR

LANGEVIN_GAMMA — friction coefficients per species

In [58]:
! cat KPOINTS

K-Spacing Value to Generate K-Mesh: 0.030
0
Gamma
  1 1 1 
0.0  0.0  0.0


...so, sampling the electron states at only one point in reciprocal space (the center). Ok for big cells.

In [75]:
! grep "VRHFIN" POTCAR
#---> chemical configurations

   VRHFIN =Sn: s2p2
   VRHFIN =I : s2p5
   VRHFIN =N: s2p3
   VRHFIN =H: ultrasoft test
   VRHFIN =C: s2p2


In [84]:
! grep "TITEL" POTCAR
#---> pseudopotential info

   TITEL  = PAW_PBE Sn_d 06Sep2000
   TITEL  = PAW_PBE I 08Apr2002
   TITEL  = PAW_PBE N 08Apr2002
   TITEL  = PAW_PBE H 15Jun2001
   TITEL  = PAW_PBE C 08Apr2002


...i.e. Sn 4d electrons treated as valence (important for accuracy).